# Bước 0: Làm sạch dữ liệu

Notebook này chạy từng bước làm sạch và hiện kết quả ngay dưới mỗi ô.
Các hàm làm sạch nằm trong `src/preprocessing.py`; notebook chỉ gọi lại để xem kết quả.

**Cách dùng:** chọn kernel `.venv` ở góc trên bên phải, rồi bấm ▶ từng ô từ trên xuống (hoặc **Run All**).

## Chuẩn bị

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
from src import preprocessing as pp

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 50)
log = []

## Đọc dữ liệu thô

Đọc mọi cột dưới dạng chữ (`dtype=str`) để giữ nguyên giá trị gốc.

In [ ]:
raw = pp.load_raw()
log.append(("Dữ liệu thô", len(raw)))
print(raw.shape)
raw.head(10)

## 0.1 Lỗi ở mức file

Dòng tiêu đề lặp, dòng rỗng, dòng bị lệch cột.

In [ ]:
print("Dòng tiêu đề lặp:", (raw["order"] == "order").sum())
print("Dòng rỗng:", (raw == "").all(axis=1).sum())
shifted = (raw["scraped_at"] == "") & raw["contact"].str.match(r"^\d{4}-\d{2}-\d{2}T|^\d{2}/\d{2}/\d{4} ")
print("Dòng lệch cột:", shifted.sum())
raw[shifted][["order", "city", "district", "house_type", "area_m2", "contact", "scraped_at"]]

In [ ]:
df = pp.fix_file_level(raw.copy(), log)
pd.DataFrame(log, columns=["Bước", "Số dòng"])

## 0.2 Chuẩn hóa chữ

Sửa lỗi font, Unicode, bỏ HTML/emoji/khoảng trắng thừa, gom ký hiệu thiếu về NaN.

In [ ]:
before = df.copy()
df = pp.normalize_text(df)

changed = before["house_type"] != df["house_type"].fillna("")
pd.DataFrame({"Trước": before.loc[changed, "house_type"], "Sau": df.loc[changed, "house_type"]}).drop_duplicates().head(15)

### Thành phố và quận

In [ ]:
print("Trước: city có", before["city"].nunique(), "giá trị, district có", before["district"].nunique(), "giá trị")
df = pp.fix_location(df)
print("Sau  : city có", df["city"].nunique(), "giá trị, district có", df["district"].nunique(), "giá trị")
df["city"].value_counts(dropna=False)

In [ ]:
pd.DataFrame({"Trước": before["district"], "Sau": df["district"]}).drop_duplicates().sample(20, random_state=1)

In [ ]:
df["location_fix"].value_counts()

## 0.3 Chuyển chữ thành số

Giá, diện tích, kích thước, số tầng, số phòng, cột có/không, ngày đăng.

In [ ]:
before = df.copy()
df = pp.parse_numbers(df)

cols = ["price_billion_vnd", "area_m2", "width_m", "floors", "bedrooms", "has_parking", "posted_date"]
sample = before[cols].apply(lambda s: ~s.astype(str).str.fullmatch(r"\d+(\.\d+)?|True|False|nan")).any(axis=1)
pd.concat({"Trước": before.loc[sample, cols], "Sau": df.loc[sample, cols]}, axis=1).sample(15, random_state=2)

In [ ]:
df[["price_billion_vnd", "area_m2", "width_m", "length_m", "road_width_m", "floors", "bedrooms"]].describe().T

## 0.4 Giá trị vô lý

Đổi số âm, 0 tầng, 99 phòng... thành NaN; sửa diện tích gõ thừa số 0; đảo ngang/dài; đối chiếu giá với tiêu đề.

In [ ]:
na_before = df.isna().sum()
df = pp.fix_invalid_values(df)
na = pd.DataFrame({"NaN trước": na_before, "NaN sau": df.isna().sum()})
na[na["NaN sau"] != na["NaN trước"]]

In [ ]:
print("Lấy giá từ tiêu đề:", df["price_from_title"].sum())
print("Giá khác tiêu đề  :", df["title_price_mismatch"].sum())
df.loc[df["title_price_mismatch"], ["title", "price_billion_vnd"]].head(10)

## 0.5 Trùng lặp

Trùng y hệt, trùng url, tin đăng lại.

In [ ]:
df = pp.drop_duplicates(df, log)
pd.DataFrame(log, columns=["Bước", "Số dòng"])

## 0.6–0.7 Kết quả cuối

Bỏ dòng không có giá, ép kiểu dữ liệu. Giá trị thiếu còn lại sẽ được điền ở Bước 2 (trong pipeline).

In [ ]:
clean = pp.finalize(df, log)
steps = pd.DataFrame(log, columns=["Bước", "Số dòng"])
steps

In [ ]:
ax = steps.plot.barh(x="Bước", y="Số dòng", legend=False, figsize=(8, 4))
ax.invert_yaxis()
ax.bar_label(ax.containers[0])
ax.set_title("Số dòng sau từng bước làm sạch")
plt.tight_layout()

In [ ]:
print(clean.shape)
print(clean.dtypes)
clean.head(10)

In [ ]:
clean.isna().sum()[lambda s: s > 0].to_frame("Số ô thiếu")

## Lưu kết quả

Ghi đè `data/processed/clean.csv` và `reports/cleaning_log.csv`.

In [ ]:
pp.CLEAN_PATH.parent.mkdir(parents=True, exist_ok=True)
pp.LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_csv(pp.CLEAN_PATH, index=False, encoding="utf-8")
steps.rename(columns={"Bước": "buoc", "Số dòng": "so_dong"}).to_csv(pp.LOG_PATH, index=False, encoding="utf-8")
print("Đã lưu", len(clean), "dòng vào", pp.CLEAN_PATH.relative_to(ROOT))